# HRS Raw Data Ingestion
# Table of Contents

1. [Document Information](#1-document-information)

2. [Purpose](#2-purpose)

3. [RAND HRS Data Acquisition](#3-rand-hrs-data-acquisition)
   * [Manual Download Process](#31-manual-download-process)
   * [Why the Download Is Manual](#32-why-the-download-is-manual)

4. [Databricks Ingestion Architecture](#4-databricks-ingestion-architecture)

5. [Landing Volume](#5-landing-volume)

6. [Bronze Layer](#6-bronze-layer)
   6.1 [Bronze Layer Purpose](#61-bronze-layer-purpose)

7. [Silver Layer](#7-silver-layer)
   * [Silver Layer Purpose](#71-silver-layer-purpose)

8. [End-to-End Ingestion Process](#8-end-to-end-ingestion-process)
   * [Step 1 — Download](#step-1--download)
   * [Step 2 — Stage](#step-2--stage)
   * [Step 3 — Validate Source](#step-3--validate-source)
   * [Step 4 — Load Bronze](#step-4--load-bronze)
   * [Step 5 — Validate Bronze](#step-5--validate-bronze)
   * [Step 6 — Transform to Silver](#step-6--transform-to-silver)
   * [Step 7 — Validate Silver](#step-7--validate-silver)

9. [Separation of Responsibilities](#9-separation-of-responsibilities)

10. [Security Considerations](#10-security-considerations)

11. [Current HRS 2022 Example](#11-current-hrs-2022-example)

12. [Summary](#12-summary)


## 1. Document Information

| Property             | Value                                                        |
| -------------------- | ------------------------------------------------------------ |
| Document Name        | RAND HRS Raw Data Ingestion                                  |
| Version              | 1.1                                                          |
| Author               | Perez                                                        |
| AI Assistant         | ChatGPT                                                      |
| Last Updated         | 2026-09-15                                                   |
| Primary Study        | Health and Retirement Study (HRS)                            |
| Primary RAND Product | RAND HRS Longitudinal File                                   |
| Study Institution    | University of Michigan                                       |
| Primary Sponsors     | National Institute on Aging / Social Security Administration |
| Primary Population   | Americans age 50 and older and their spouses                 |
| Survey Frequency     | Approximately every two years                                |
| Primary Uses         | Aging, health, retirement, economic, and policy research     |

---

## 2. Purpose

This document describes the process used to acquire the **RAND HRS Longitudinal File** and ingest it into the Databricks HRS data platform.

The process has two distinct stages:

1. **Manual data acquisition** — The authorized user logs into the HRS website and downloads the RAND HRS data file.
2. **Automated Databricks ingestion** — The downloaded file is placed in the Databricks Landing Volume and processed into the Bronze and Silver data layers.

This approach keeps the external RAND authentication process separate from the automated Databricks ETL pipeline.

---

## 3. RAND HRS Data Acquisition

The RAND HRS Longitudinal File is distributed through the Health and Retirement Study website at the University of Michigan.

Users must have an authorized HRS account and log in before downloading restricted-access data products.

The RAND HRS download packages are provided as compressed files containing statistical data formats such as:

* SAS
* Stata
* SPSS

### 3.1 Manual Download Process

The data acquisition process is:

1. Navigate to the RAND HRS Longitudinal File product page.
2. Log in using your authorized HRS account.
3. Locate the required RAND HRS release.
4. Download the required data package.
5. Save the downloaded ZIP file to a local location.
6. Upload the required data file to the Databricks Landing Volume.

### 3.2 Why the Download Is Manual

The HRS website uses web-based authentication and security controls. Automated HTTP requests from a Databricks Python process are blocked by the site's Cloudflare security layer.

Because of this, the ingestion process **does not attempt to automate the RAND website login**.

This is intentional.

The Databricks ETL process begins **after the authorized user has downloaded the RAND HRS file**.

---

# 4. Databricks Ingestion Architecture

Once the RAND HRS file has been downloaded, the remainder of the ingestion process is performed in Databricks.

The overall architecture is:

```text
┌───────────────────────────────┐
│       RAND HRS Website        │
│                               │
│  1. Log in                    │
│  2. Download RAND HRS file    │
└───────────────┬───────────────┘
                │
                │ Manual Download
                ▼
┌───────────────────────────────┐
│      Databricks Landing       │
│          Volume               │
│                               │
│ landing_catalog               │
│   external_data               │
│     rand_hrs_raw_data         │
└───────────────┬───────────────┘
                │
                │ Automated ETL
                ▼
┌───────────────────────────────┐
│         Bronze Layer          │
│                               │
│ dev_catalog                   │
│   brz_raw_hrs                 │
│                               │
│ randhrs1992_2022v1            │
└───────────────┬───────────────┘
                │
                │ Transform
                ▼
┌───────────────────────────────┐
│         Silver Layer          │
│                               │
│ dev_catalog                   │
│   slv_cdm_hrs                 │
│                               │
│ hub_respondent                │
│ dim_wave                      │
│ Demographics                  │
│ Health                        │
│ Financial                     │
│ Employment                    │
│ ...                           │
└───────────────────────────────┘
```

---

# 5. Landing Volume

The downloaded RAND HRS file is first placed in the Databricks Landing Volume.

The project uses:

```text
/Volumes/landing_catalog/external_data/rand_hrs_raw_data/
```

The Landing Volume serves as the **raw file staging area**.

The source file should be retained in its original format whenever practical. This provides an original source copy that can be used for:

* Reprocessing
* Troubleshooting
* Data validation
* Comparing releases
* Rebuilding Bronze tables

### Example

For the 2022 RAND HRS Longitudinal File, the source data file is:

```text
randhrs1992_2022v1.sav
```

and is stored in:

```text
/Volumes/landing_catalog/external_data/rand_hrs_raw_data/
```

---

# 6. Bronze Layer

The Bronze layer contains the RAND HRS data after it has been converted from the original statistical file format into a Databricks Delta table.

The target catalog and schema are:

```text
dev_catalog.brz_raw_hrs
```

The 2022 longitudinal file is loaded into:

```text
dev_catalog.brz_raw_hrs.randhrs1992_2022v1
```

### Bronze Layer Purpose

The Bronze table provides a Databricks-native representation of the original RAND HRS data while preserving the source data as closely as practical.

The Bronze layer is primarily intended for:

* Raw data retention
* Source-level analysis
* Data quality validation
* Downstream transformation
* Reprocessing
* Auditing and troubleshooting

The Bronze layer should generally **not** contain business-specific transformations that change the meaning of the source data.

---

# 7. Silver Layer

The Silver layer transforms the Bronze data into the project's standardized **HRS Common Data Model (CDM)**.

The target catalog and schema are:

```text
dev_catalog.slv_cdm_hrs
```

Examples of Silver CDM tables include:

```text
hub_respondent
dim_wave
hrs_demographics
hrs_health
hrs_financial
hrs_employment
...
```

The exact tables depend on the HRS survey sections being implemented.

### Silver Layer Purpose

The Silver layer provides a structured analytical model that is easier to use for:

* Data analysis
* SQL queries
* Reporting
* Visualization
* Statistical analysis
* Research
* Public-policy analysis

The Silver layer also establishes consistent relationships between respondents, waves, and survey-specific observations.

---

# 8. End-to-End Ingestion Process

The complete ingestion process is:

### Step 1 — Download

The authorized user downloads the required RAND HRS release from the HRS website.

### Step 2 — Stage

The downloaded file is placed in:

```text
/Volumes/landing_catalog/external_data/rand_hrs_raw_data/
```

### Step 3 — Validate Source

The ingestion process verifies that:

* The expected file exists.
* The file can be read.
* The expected source format is present.
* The expected RAND HRS release is being processed.

### Step 4 — Load Bronze

The source file is converted into a Delta table:

```text
dev_catalog.brz_raw_hrs.randhrs1992_2022v1
```

### Step 5 — Validate Bronze

Data-quality checks are performed to verify that:

* The table was created successfully.
* Rows were loaded.
* Expected columns exist.
* Key source variables are populated as expected.
* The resulting data is consistent with the source file.

### Step 6 — Transform to Silver

The Bronze data is transformed according to the applicable HRS Silver CDM DDL and DML specifications.

### Step 7 — Validate Silver

The resulting Silver tables are validated for:

* Referential integrity
* Required keys
* Expected row counts
* Valid respondent and wave relationships
* Data completeness
* Transformation accuracy

---

# 9. Separation of Responsibilities

The architecture intentionally separates **data acquisition** from **data engineering**.

| Activity                            | Responsibility                            |
| ----------------------------------- | ----------------------------------------- |
| HRS account registration            | Authorized user                           |
| HRS website authentication          | Authorized user                           |
| RAND HRS file download              | Authorized user                           |
| Upload to Databricks Landing Volume | Data engineer / authorized user           |
| Source validation                   | Databricks ETL                            |
| Bronze table creation               | Databricks ETL                            |
| Bronze data validation              | Databricks ETL                            |
| Silver transformation               | Databricks ETL                            |
| Silver CDM validation               | Databricks ETL                            |
| CI/CD deployment                    | GitHub Actions / Databricks Asset Bundles |

This separation prevents external website authentication from becoming a dependency of the Databricks ETL pipeline.

---

# 10. Security Considerations

RAND HRS data should be handled according to the applicable HRS data-use requirements and the project's security policies.

The ingestion process should follow these principles:

* Do not store RAND passwords in notebooks.
* Do not commit RAND credentials to GitHub.
* Do not place credentials in SQL files.
* Do not place credentials in Asset Bundle configuration files.
* Do not include credentials in source-code comments or documentation.
* Retain the original downloaded source file in the designated Landing location.
* Limit access to the Landing, Bronze, and Silver data according to project requirements.

Because RAND website authentication is performed manually, RAND credentials are **not required by the automated Databricks ETL process**.

---

# 11. Current HRS 2022 Example

The current implementation uses the RAND HRS Longitudinal File covering the 1992–2022 period.

### Source

```text
RAND HRS Longitudinal File 2022
randhrs1992_2022v1.sav
```

### Landing

```text
/Volumes/landing_catalog/external_data/rand_hrs_raw_data/
```

### Bronze

```text
dev_catalog.brz_raw_hrs.randhrs1992_2022v1
```

### Silver

```text
dev_catalog.slv_cdm_hrs
```

The Silver CDM is expanded incrementally as additional HRS survey sections are implemented.

---

# 12. Summary

The HRS Raw Data Ingestion architecture provides a clear boundary between **external data acquisition** and **automated data engineering**.

The authorized user performs the RAND HRS download through the HRS website. Once the file is placed in the Databricks Landing Volume, the remaining ingestion process can be automated through Databricks.

The resulting architecture is:

```text
RAND HRS
   │
   │ Manual authorized download
   ▼
Landing Volume
   │
   │ Automated
   ▼
Bronze Delta
   │
   │ Transform
   ▼
Silver HRS CDM
   │
   ▼
Analytics / Reporting / Research
```

This approach provides a practical, secure, and maintainable method for incorporating RAND HRS data into the Databricks HRS data platform while keeping external authentication separate from the automated ETL pipeline.
